# 07 — Extracción de los capítulos 5 y 6: causas por tipo de proceso
525 páginas y ~11.700 filas. Aportan la variable que no está en ninguna otra parte: el **tipo
de proceso**. La unidad es (ámbito, ciudad o distrito, materia, tipo de proceso); el anuario
no tiene juzgados individuales.

Se trabaja con coordenadas reales (`pdftotext -bbox-layout`) por dos motivos:
1. Las etiquetas de grupo (ORDINARIO, MONITOREO...) están impresas **en vertical** en el margen.
2. Los rótulos largos se parten en varias líneas **con los números en el medio**; pdftotext
   agrupa esas líneas en un solo `<block>` porque son una sola celda del PDF.

Cada página se reconoce por su **firma de encabezado** y su orden de columnas sale de
`LAYOUTS_PROCESOS` (módulo procesos). Los valores salen como texto crudo.

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/geografia.ipynb
%run -i modulos/procesos.ipynb
import hashlib
import unicodedata
import matplotlib.pyplot as plt

# Número: 1.234, 45, 70,7%, -3. Sin punto final, para que el "5.1.1." del membrete no cuente.
RE_NUMERO = re.compile(r"^-?\d(?:[\d.,]*\d)?%?$")
RE_DESCARTE = re.compile(r"^\s*(FUENTE|Fuente|NOTA|Nota|ELABORACI|Elaboraci|\(\d+\)|Contin|Cuadro|CUADRO|Gesti[oó]n|GESTI[OÓ]N)")
# Entre palabras de un rótulo hay 1,3 a 3,0 puntos ("Art." y "207" en la p. 177 están a 1,4);
# del rótulo a la primera columna hay decenas.
HUECO_ROTULO = 3.0
FUERA = {"DE", "DEL", "LA", "EL", "LOS", "LAS", "EN", "Y", "O", "A", "POR", "CON", "N", "AL", "ES", "SU"}

PALABRAS_ENTIDAD = {"NACIONAL"}
for e in ENTIDADES:
    for p in e.split():
        PALABRAS_ENTIDAD.add(p)

problemas = []


def anotar(cuadro, pagina, motivo, detalle=""):
    problemas.append({"cuadro_origen": cuadro, "pagina_pdf": pagina, "motivo": motivo,
                      "detalle": " ".join(str(detalle).split())[:300]})


def guardar(df, nombre):
    df.to_csv(INTERIM / nombre, index=False, encoding="utf-8")
    print(nombre, len(df), "filas")

## Texto

In [ ]:
def sin_tildes(texto):
    t = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return " ".join(re.sub(r"[^A-Z0-9]+", " ", t.upper()).split())


def normalizar(texto):
    # sin_tildes y sin palabras de enlace
    palabras = []
    for p in sin_tildes(texto).split():
        if p not in FUERA:
            palabras.append(p)
    return " ".join(palabras)


def es_entidad(texto):
    # "SUCRE", "LA PAZ 30", "TOTAL NACIONAL 153". Sin quitar artículos: "LA PAZ" los necesita.
    return sin_tildes(re.sub(r"(\s+\d[\d.,]*)+$", "", texto.strip())) in ENTIDADES


def resolver_nombre_entidad(etiquetas):
    # La única entidad válida entre los fragmentos del renglón (en la p. 362 aparece "L APAZ" junto a "LA PAZ").
    candidatas = set()
    for t in etiquetas:
        if sin_tildes(t) in ENTIDADES:
            candidatas.add(sin_tildes(t))
    if len(candidatas) == 1:
        return candidatas.pop()
    return None


def descartable(texto):
    return es_ruido(texto) or RE_DESCARTE.match(texto) is not None


def centro_de_grupo(grupo):
    total = 0
    for w in grupo:
        total = total + (w[1] + w[3]) / 2
    return total / len(grupo)

## Geometría

In [ ]:
def es_rotada(palabra):
    # Vertical: el alto supera 2,2 veces el ancho. Con 1,5 los dígitos de cuerpo grande
    # (4 x 7,5 en las páginas 298 y 505) se tomaban por texto rotado.
    x0 = palabra[0]
    y0 = palabra[1]
    x1 = palabra[2]
    y1 = palabra[3]
    return (y1 - y0) > (x1 - x0) * 2.2 and (y1 - y0) >= 7.5 and len(palabra[4].strip()) >= 3


def unir_filas_partidas(grupos):
    # Una fila con rótulo de dos o tres líneas centra el último valor y queda partida en dos.
    # Se vuelve a pegar si los pedazos están cerca, ocupan columnas distintas y juntos suman
    # el ancho de fila más frecuente de la página (que ninguno alcanza solo).
    if len(grupos) == 0:
        return []
    anchos = []
    for g in grupos:
        anchos.append(len(g))
    modal = max(set(anchos), key=anchos.count)
    centros = []
    for g in grupos:
        centros.append(centro_de_grupo(g))
    saltos = []
    for i in range(1, len(centros)):
        saltos.append(centros[i] - centros[i - 1])
    saltos.sort()
    paso = 0
    if len(saltos) > 0:
        paso = saltos[len(saltos) // 2]

    unidos = [list(grupos[0])]
    for g in grupos[1:]:
        previo = unidos[-1]
        cerca = centro_de_grupo(g) - centro_de_grupo(previo) < 0.75 * paso
        completa = (len(previo) + len(g) == modal) and (modal != len(previo)) and (len(g) != modal)
        disjuntos = True
        for a in previo:
            for b in g:
                if min(a[2], b[2]) - max(a[0], b[0]) > 0:
                    disjuntos = False
        if cerca and completa and disjuntos:
            previo.extend(g)
            previo.sort(key=lambda w: w[0])
        else:
            unidos.append(list(g))
    return unidos


def numeros_del_rotulo(palabras):
    # Números que son parte del rótulo: llamadas al pie ("Yapacani1") y artículos ("Art. 207").
    # Un número pegado (hueco < 3 puntos) a un texto, o a otro número del rótulo, es rótulo.
    del_rotulo = set()
    for i in range(len(palabras)):
        w = palabras[i]
        if not RE_NUMERO.match(w[4]):
            continue
        vecinos = []
        if i > 0:
            vecinos.append((i - 1, w[0] - palabras[i - 1][2]))
        else:
            vecinos.append((i - 1, None))
        if i + 1 < len(palabras):
            vecinos.append((i + 1, palabras[i + 1][0] - w[2]))
        else:
            vecinos.append((i + 1, None))
        for j, hueco in vecinos:
            if hueco is None or hueco >= HUECO_ROTULO:
                continue
            if not RE_NUMERO.match(palabras[j][4]) or j in del_rotulo:
                del_rotulo.add(i)
                break
    # Un número del rótulo arrastra a los números pegados a él ("ART. 415 Y SGTES."); se repite.
    while True:
        nuevos = set()
        for i in range(len(palabras)):
            w = palabras[i]
            if i in del_rotulo or not RE_NUMERO.match(w[4]):
                continue
            pegado_izq = i > 0 and (i - 1) in del_rotulo and w[0] - palabras[i - 1][2] < HUECO_ROTULO
            pegado_der = i + 1 < len(palabras) and (i + 1) in del_rotulo and palabras[i + 1][0] - w[2] < HUECO_ROTULO
            if pegado_izq or pegado_der:
                nuevos.add(i)
        if len(nuevos) == 0:
            return del_rotulo
        del_rotulo = del_rotulo | nuevos


def armar_linea(palabras):
    caja = caja_de(palabras)
    texto = " ".join(w[4] for w in palabras)
    return (caja[0], caja[1], caja[2], caja[3], texto, palabras)


def etiquetas_rotadas(lineas):
    # Junta el texto vertical del margen en etiquetas de grupo. En texto rotado el ancho de la
    # caja es el cuerpo de letra y cada etiqueta tiene el suyo: así se separan dos etiquetas
    # pegadas (fin de ORDINARIO y principio de EXTRAORDINARIO). El texto queda verbatim
    # ("EXTRAORDI NARIO"); se limpia en el paso 04.
    tiras = []
    for linea in lineas:
        actual = []
        for w in sorted(linea, key=lambda w: -w[1]):
            if len(actual) > 0:
                ancho_a = actual[-1][2] - actual[-1][0]
                ancho_w = w[2] - w[0]
                if abs(ancho_a - ancho_w) > 0.2 * max(ancho_a, ancho_w):
                    tiras.append(actual)
                    actual = []
            actual.append(w)
        if len(actual) > 0:
            tiras.append(actual)

    etiquetas = []
    for tira in sorted(tiras, key=lambda t: min(w[0] for w in t)):
        caja = list(caja_de(tira))
        fusionada = False
        for e in etiquetas:
            if caja[0] - e[2] < 4 and min(caja[3], e[3]) - max(caja[1], e[1]) > 0:
                e[1] = min(e[1], caja[1])
                e[2] = max(e[2], caja[2])
                e[3] = max(e[3], caja[3])
                e[4].append(tira)
                fusionada = True
                break
        if not fusionada:
            etiquetas.append([caja[0], caja[1], caja[2], caja[3], [tira]])

    hechas = []
    for e in sorted(etiquetas, key=lambda e: e[1]):
        palabras = []
        for tira in e[4]:
            for w in tira:
                palabras.append(w[4])
        hechas.append([e[0], e[1], e[2], e[3], " ".join(palabras)])
    # Un fragmento de una o dos letras es el resto de la etiqueta vecina ("PROCES" y "O", p. 132).
    sueltas = []
    enteras = []
    for e in hechas:
        if len(e[4].replace(" ", "")) <= 2:
            sueltas.append(e)
        else:
            enteras.append(e)
    for e in sueltas:
        if len(enteras) == 0:
            enteras.append(e)
            continue
        centro = (e[1] + e[3]) / 2
        v = min(enteras, key=lambda o: abs((o[1] + o[3]) / 2 - centro))
        v[1] = min(v[1], e[1])
        v[3] = max(v[3], e[3])
        v[0] = min(v[0], e[0])
        v[2] = max(v[2], e[2])
        if e[1] > v[1]:
            v[4] = v[4] + " " + e[4]
        else:
            v[4] = e[4] + " " + v[4]
    salida = []
    for e in sorted(enteras, key=lambda e: e[1]):
        salida.append(tuple(e))
    return salida

## Lectura de una página
Devuelve un diccionario con las filas clasificadas (encabezado, entidad, datos, suelta),
las etiquetas de grupo, los límites de columna y la firma de encabezado.

In [ ]:
def solape_vertical(caja, fila):
    return min(caja[3], fila["y_fin"]) - max(caja[1], fila["y_ini"])


def armar_filas(pag):
    # Las filas se arman desde los números (agrupados por centro vertical) y el rótulo se pega después.
    grupos = unir_filas_partidas(agrupar_por_y(pag["numericos"]))
    filas = []
    for grupo in grupos:
        filas.append({"y": centro_de_grupo(grupo), "numeros": grupo, "x_num": min(w[0] for w in grupo),
                      "y_ini": min(w[1] for w in grupo), "y_fin": max(w[3] for w in grupo),
                      "piezas": [], "x_rotulo": None, "clase": None})

    # Un bloque de texto que toca una sola fila es su rótulo entero. Si toca varias
    # (p. 304: la ciudad y trece rótulos en un bloque), cada línea va a la fila más cercana.
    for b in pag["textos"]:
        dentro = []
        for f in filas:
            if solape_vertical(b, f) > 0:
                dentro.append(f)
        if len(dentro) == 0:
            continue
        if len(dentro) == 1 and b[2] < dentro[0]["x_num"]:
            dentro[0]["piezas"].append((b[1], b[0], b[2], b[4]))
            continue
        for l in b[5]:
            cabe = []
            for f in dentro:
                if l[2] < f["x_num"]:
                    cabe.append(f)
            if len(cabe) > 0:
                centro_l = (l[1] + l[3]) / 2
                destino = max(cabe, key=lambda f: (solape_vertical(l, f), -abs(f["y"] - centro_l)))
                destino["piezas"].append((l[1], l[0], l[2], l[4]))

    for f in filas:
        textos = []
        etiquetas = set()
        for pieza in f["piezas"]:
            textos.append(pieza[3])
            etiquetas.add(pieza[3])
        f["rotulo"] = " ".join(textos)
        if len(f["piezas"]) > 0:
            f["x_rotulo"] = min(p[1] for p in f["piezas"])
        else:
            f["x_rotulo"] = None
        f["nombre_entidad"] = resolver_nombre_entidad(etiquetas)

    # Cabeceras de entidad sin número de juzgados: no tienen números, se buscan entre los bloques sobrantes.
    usados = set()
    for f in filas:
        for pieza in f["piezas"]:
            usados.add(pieza[3])
    for b in pag["textos"]:
        if b[4] not in usados and sin_tildes(b[4]) in ENTIDADES:
            filas.append({"y": (b[1] + b[3]) / 2, "numeros": [], "piezas": [], "x_num": 10 ** 6,
                          "x_rotulo": b[0], "rotulo": b[4], "nombre_entidad": sin_tildes(b[4]), "clase": None})
    filas.sort(key=lambda f: f["y"])

    # La tabla empieza en la primera fila que es una entidad; lo de arriba es encabezado.
    inicio = None
    for i in range(len(filas)):
        if filas[i]["nombre_entidad"]:
            inicio = i
            break
    if inicio is None:
        pag["y_tabla"] = None
        cuerpo = filas
    else:
        pag["y_tabla"] = filas[inicio]["y"]
        cuerpo = filas[inicio:]
        for f in filas[:inicio]:
            f["clase"] = "encabezado"

    # Columna angosta de rótulos a la izquierda (grupo escrito en horizontal, p. ej. en 6.1.2.7):
    # sale del rótulo y se trata como las etiquetas rotadas.
    for f in filas:
        f["piezas"].sort()
    anchos = []
    for f in filas:
        if len(f["numeros"]) >= 2 and len(f["piezas"]) > 0:
            anchos.append(min(p[1] for p in f["piezas"]))
    x_principal = 0
    if len(anchos) > 0:
        x_principal = max(set(anchos), key=anchos.count)
    horizontales = []
    for f in filas:
        izq = []
        for p in f["piezas"]:
            if p[2] < x_principal - 5 and sin_tildes(p[3]) not in ENTIDADES:
                izq.append(p)
        if len(izq) > 0 and len(izq) < len(f["piezas"]):
            resto = []
            for p in f["piezas"]:
                if p not in izq:
                    resto.append(p)
            f["piezas"] = resto
            horizontales = horizontales + izq

    # Texto vertical arriba de la tabla = rótulo de columna; dentro de la tabla = etiqueta de grupo.
    y_corte = 0
    if pag["y_tabla"] is not None:
        y_corte = pag["y_tabla"]
    pag["rotadas_encabezado"] = []
    pag["etiquetas_grupo"] = []
    for e in pag["rotadas"]:
        if e[3] <= y_corte:
            pag["rotadas_encabezado"].append(e)
        else:
            pag["etiquetas_grupo"].append(e)
    for y, x0, x1, t in sorted(set(horizontales)):
        pag["etiquetas_grupo"].append((x0, y, x1, y, t))

    # Límites provisionales con las filas que seguro son de datos (3+ números).
    anchas = []
    for f in cuerpo:
        if len(f["numeros"]) >= 3:
            anchas.append(f)
    if len(anchas) == 0:
        for f in cuerpo:
            if len(f["numeros"]) >= 2:
                anchas.append(f)
    rangos = []
    for f in anchas:
        for w in f["numeros"]:
            rangos.append((w[0], w[2]))
    provisorios = agrupar_por_x(rangos)

    for f in cuerpo:
        fuera = []
        for w in f["numeros"]:
            adentro = False
            for a, b in provisorios:
                if min(w[2], b) - max(w[0], a) > 0:
                    adentro = True
            if not adentro:
                fuera.append(w)
        if f["nombre_entidad"] and (len(f["numeros"]) == 0 or len(fuera) > 0):
            # Cabecera de ciudad: su número de juzgados cae a la izquierda de las columnas de datos.
            f["clase"] = "entidad"
            f["numeros"] = fuera
        elif len(f["numeros"]) >= 2 and f["rotulo"]:
            f["clase"] = "datos"
        else:
            f["clase"] = "suelta"
    return filas


def armar_encabezado(pag, y_membrete):
    # Banda de encabezado: del membrete del cuadro a la primera fila, sin la línea de la entidad.
    # La firma es la lista ordenada de palabras normalizadas de la banda.
    if pag["y_tabla"] is not None:
        y_primera = pag["y_tabla"]
    elif len(pag["filas"]) > 0:
        y_primera = min(f["y"] for f in pag["filas"])
    else:
        y_primera = 10 ** 6
    banda = []
    for b in pag["textos"]:
        for l in b[5]:
            if y_membrete <= l[1] and l[3] < y_primera:
                banda.append(l)
    for w in pag["numericos"]:
        if y_membrete <= w[1] and w[3] < y_primera:
            banda.append(w)
    for e in pag["rotadas_encabezado"]:
        if y_membrete <= e[1]:
            banda.append(e)
    filas_banda = agrupar_por_y(banda)
    while len(filas_banda) > 0 and es_entidad(" ".join(c[4] for c in filas_banda[-1])):
        filas_banda.pop()
    celdas = []
    for fila in filas_banda:
        for c in fila:
            celdas.append(c)
    toks = []
    for c in celdas:
        for t in normalizar(c[4]).split():
            if not t.isdigit() and t not in PALABRAS_ENTIDAD:
                toks.append(t)
    toks.sort()
    return " ".join(toks), celdas


def leer_pagina(numero, bloques):
    pag = {"numero": numero}
    # Texto vertical: se detecta por palabra (pdftotext mezcla etiquetas rotadas en un bloque).
    rotadas = []
    limpios = []
    for b in bloques:
        lineas = []
        for l in b[5]:
            rot = []
            resto = []
            for w in l[5]:
                if es_rotada(w):
                    rot.append(w)
                else:
                    resto.append(w)
            if len(rot) > 0:
                rotadas.append(rot)
            if len(resto) > 0:
                lineas.append(armar_linea(resto))
        if len(lineas) > 0:
            limpios.append(armar_linea(lineas))
    pag["rotadas"] = etiquetas_rotadas(rotadas)

    # Números y texto se separan por palabra: lo único estable entre páginas.
    numericos = []
    textos = []
    for b in limpios:
        if descartable(b[4]):
            continue
        lineas = []
        for l in b[5]:
            del_rotulo = numeros_del_rotulo(l[5])
            num = []
            for i in range(len(l[5])):
                if RE_NUMERO.match(l[5][i][4]) and i not in del_rotulo:
                    num.append(l[5][i])
            numericos = numericos + num
            resto = []
            for w in l[5]:
                if w not in num:
                    resto.append(w)
            if len(resto) > 0:
                lineas.append(armar_linea(resto))
        if len(lineas) > 0:
            textos.append(armar_linea(lineas))
    pag["numericos"] = numericos
    pag["textos"] = textos

    # Título de la página ("Juzgados Públicos en Materia Civil..."): lo que está arriba del membrete.
    y_membrete = 0
    ys = []
    for b in limpios:
        if RE_DESCARTE.match(b[4]) and (b[4].startswith("Cuadro") or b[4].startswith("CUADRO")):
            ys.append(b[1])
    if len(ys) > 0:
        y_membrete = min(ys)
    arriba = []
    for b in limpios:
        if b[3] <= y_membrete and not es_ruido(b[4]):
            arriba.append(b)
    arriba.sort(key=lambda b: b[1])
    pag["titulo_pagina"] = " ".join(b[4] for b in arriba)

    pag["filas"] = armar_filas(pag)
    rangos = []
    for f in pag["filas"]:
        if f["clase"] == "datos":
            for w in f["numeros"]:
                rangos.append((w[0], w[2]))
    pag["limites"] = agrupar_por_x(rangos)
    pag["firma"], pag["encabezado"] = armar_encabezado(pag, y_membrete)
    return pag


def firma_corta(firma):
    return hashlib.sha1(firma.encode("utf-8")).hexdigest()[:10]

## Leer las 525 páginas y agruparlas por firma

In [ ]:
catalogo = pd.read_csv(INTERIM / "catalogo_cuadros.csv")
catalogo = catalogo[catalogo["cuadro_id"].notna()]
catalogo = catalogo[catalogo["cuadro_id"].astype(str).str.startswith(("5.", "6."))]
catalogo = catalogo.sort_values("pagina_pdf")
# página -> número de cuadro tal como lo imprime el PDF (con sus errores de numeración)
inv = {}
for i, r in catalogo.iterrows():
    inv[int(r["pagina_pdf"])] = str(r["cuadro_id"])

crudo = bloques_bbox(min(inv), max(inv))
paginas_leidas = {}
for p in inv:
    pag = leer_pagina(p, crudo[p])
    tiene_datos = False
    for f in pag["filas"]:
        if f["clase"] == "datos":
            tiene_datos = True
    if not tiene_datos:
        anotar(inv[p], p, "sin filas de datos reconocibles")
        continue
    paginas_leidas[p] = pag

grupos_firma = {}
for p in paginas_leidas:
    firma = paginas_leidas[p]["firma"]
    if firma not in grupos_firma:
        grupos_firma[firma] = []
    grupos_firma[firma].append(p)

# Para cada firma: número de columnas más frecuente y una página de referencia que lo tenga.
formatos = {}
for firma in grupos_firma:
    pags_firma = grupos_firma[firma]
    cuenta = {}
    for p in pags_firma:
        n = len(paginas_leidas[p]["limites"])
        cuenta[n] = cuenta.get(n, 0) + 1
    n = max(cuenta, key=lambda k: (cuenta[k], k))
    referencia = None
    for p in pags_firma:
        if referencia is None and len(paginas_leidas[p]["limites"]) == n:
            referencia = p
    cuadros = set()
    for p in pags_firma:
        cuadros.add(inv[p])
    formatos[firma] = {"clave": firma_corta(firma), "paginas": pags_firma, "n_columnas": n,
                       "referencia": referencia, "cuadros": sorted(cuadros)}
print(len(paginas_leidas), "páginas leídas,", len(formatos), "firmas de encabezado distintas")

## Planilla de firmas: el rótulo que imprime el PDF sobre cada columna
Es la planilla desde la que se escribieron los nombres de `LAYOUTS_PROCESOS`; el paso 04 la
usa para poner el rótulo literal al lado de cada valor.

In [ ]:
filas = []
for firma, fmt in sorted(formatos.items(), key=lambda kv: kv[1]["referencia"]):
    ref = paginas_leidas[fmt["referencia"]]
    textos = []
    for i in range(fmt["n_columnas"]):
        textos.append([])
    for celda in sorted(ref["encabezado"], key=lambda c: (c[1], c[0])):
        for i in range(len(ref["limites"])):
            a = ref["limites"][i][0]
            b = ref["limites"][i][1]
            if min(celda[2], b) - max(celda[0], a) > 0:
                textos[i].append(celda[4])
    for i in range(len(textos)):
        filas.append({
            "firma": fmt["clave"],
            "cuadros": ",".join(fmt["cuadros"]),
            "paginas": str(min(fmt["paginas"])) + "-" + str(max(fmt["paginas"])),
            "n_paginas": len(fmt["paginas"]),
            "pagina_referencia": fmt["referencia"],
            "n_columnas": fmt["n_columnas"],
            "columna": i + 1,
            "encabezado_pdf": " / ".join(textos[i]),
            "titulo_pagina": ref["titulo_pagina"],
        })
firmas = pd.DataFrame(filas)
guardar(firmas, "firmas_procesos.csv")
firmas.head(10)

## Columnas y grupos

In [ ]:
def cortes(limites):
    # Frontera entre columnas: punto medio entre una y la siguiente.
    salida = []
    for i in range(1, len(limites)):
        salida.append((limites[i - 1][1] + limites[i][0]) / 2)
    return salida


def posiciones_unicas(celdas):
    cuenta = {}
    pos = {}
    for c in celdas:
        t = normalizar(c[4])
        cuenta[t] = cuenta.get(t, 0) + 1
        pos[t] = c[0]
    salida = {}
    for t in pos:
        if cuenta[t] == 1 and t:
            salida[t] = pos[t]
    return salida


def desplazamiento(pagina, referencia):
    # Cuánto está corrida la página respecto de la referencia (hasta 20 puntos entre las
    # pp. 375 y 378): mediana de la diferencia de posición de los rótulos de encabezado.
    a = posiciones_unicas(pagina["encabezado"])
    b = posiciones_unicas(referencia["encabezado"])
    difs = []
    for t in a:
        if t in b:
            difs.append(a[t] - b[t])
    difs.sort()
    if len(difs) == 0:
        return 0.0
    return difs[len(difs) // 2]


def asignar(fila, cortes_x, n):
    # Cada número va a su columna según el centro de su caja.
    valores = [None] * n
    choques = []
    for w in fila["numeros"]:
        centro = (w[0] + w[2]) / 2
        i = 0
        while i < len(cortes_x) and centro > cortes_x[i]:
            i = i + 1
        if valores[i] is None:
            valores[i] = w[4]
        else:
            choques.append(w[4])
    return valores, choques


def asignar_grupos(filas, etiquetas):
    # Cada etiqueta de grupo está centrada en SU grupo de filas. Se busca el corte en tramos
    # consecutivos que minimiza la distancia entre el centro de cada tramo y su etiqueta
    # (programación dinámica; en la p. 121 el ajuste da menos de 2,5 puntos por grupo).
    n = len(filas)
    k = len(etiquetas)
    if n == 0 or k == 0 or k > n:
        return [None] * n
    acum = [0.0]
    for f in filas:
        acum.append(acum[-1] + f["y"])
    centros = []
    for e in etiquetas:
        centros.append((e[1] + e[3]) / 2)
    INF = float("inf")
    mejor = []
    corte = []
    for j in range(n + 1):
        mejor.append([INF] * (k + 1))
        corte.append([0] * (k + 1))
    mejor[0][0] = 0.0
    for j in range(1, n + 1):
        for g in range(1, k + 1):
            for i in range(g - 1, j):
                if mejor[i][g - 1] == INF:
                    continue
                costo = abs((acum[j] - acum[i]) / (j - i) - centros[g - 1])
                c = mejor[i][g - 1] + costo
                if c < mejor[j][g]:
                    mejor[j][g] = c
                    corte[j][g] = i
    if mejor[n][k] == INF:
        return [None] * n
    grupos = [None] * n
    j = n
    for g in range(k, 0, -1):
        i = corte[j][g]
        for t in range(i, j):
            grupos[t] = etiquetas[g - 1][4]
        j = i
    return grupos

## Filas de una página
Tres formas conviven: (a) cabecera "SUCRE 14" y abajo una fila por tipo de proceso; (b) una
fila por ciudad, sin tipo de proceso; (c) mixta: la fila de la ciudad trae juzgados y totales
y abajo van sus filas de detalle.

In [ ]:
def cerrar_bloque(bloque, salida, pagina, x_grupo):
    # Asigna las etiquetas de grupo a las filas de la entidad que termina.
    if len(bloque) == 0:
        return
    y0 = bloque[0]["fila"]["y"]
    y1 = bloque[-1]["fila"]["y"]
    etiquetas = []
    for e in pagina["etiquetas_grupo"]:
        if y0 - 12 <= (e[1] + e[3]) / 2 <= y1 + 12:
            etiquetas.append(e)
    agrupables = []
    for f in bloque:
        x = f["fila"]["x_rotulo"]
        if not x:
            x = 10 ** 6
        if x < x_grupo + 2:
            agrupables.append(f)
    filas_agrupables = []
    for f in agrupables:
        filas_agrupables.append(f["fila"])
    grupos = asignar_grupos(filas_agrupables, etiquetas)
    for i in range(len(agrupables)):
        agrupables[i]["grupo_proceso"] = grupos[i]
    salida.extend(bloque)
    bloque.clear()


def filas_de_pagina(pagina, cuadro, fmt, familia, nombres, cortes_x, n):
    entidades_pagina = []
    for f in pagina["filas"]:
        if f["nombre_entidad"]:
            entidades_pagina.append(f["nombre_entidad"])
    ambito = ambito_de_contexto(pagina["titulo_pagina"], entidades_pagina)
    if ambito is None:
        anotar(cuadro, pagina["numero"], "no se pudo derivar el ámbito desde el encabezado ni las entidades")

    datos = []
    for f in pagina["filas"]:
        if f["clase"] == "datos":
            datos.append(f)
    con_nombre = []
    sin_nombre = []
    for f in datos:
        if f["nombre_entidad"]:
            con_nombre.append(f)
        elif not f["rotulo"].upper().startswith("TOTAL"):
            sin_nombre.append(f)
    hay_cabecera = False
    for f in pagina["filas"]:
        if f["clase"] == "entidad":
            hay_cabecera = True
    por_fila = len(con_nombre) > 0 and len(sin_nombre) == 0
    mixta = len(con_nombre) > 0 and len(sin_nombre) > 0 and not hay_cabecera
    x_grupo = 0
    xs = []
    for f in datos:
        if f["x_rotulo"] is not None:
            xs.append(f["x_rotulo"])
    if len(xs) > 0:
        x_grupo = min(xs)

    entidad = None
    juzgados = None
    salida = []
    bloque = []
    for fila in pagina["filas"]:
        if fila["clase"] == "entidad":
            cerrar_bloque(bloque, salida, pagina, x_grupo)
            entidad = fila["nombre_entidad"]
            if not entidad:
                entidad = " ".join(fila["rotulo"].split())
            juzgados = None
            if len(fila["numeros"]) > 0:
                juzgados = fila["numeros"][0][4]
            continue
        if fila["clase"] != "datos":
            continue
        etiqueta = " ".join(fila["rotulo"].split())
        if mixta and (fila["nombre_entidad"] or etiqueta.upper() in ("TOTAL", "TOTAL NACIONAL")):
            # La fila de la ciudad abre su sección y además es su total: se emite como fila.
            cerrar_bloque(bloque, salida, pagina, x_grupo)
            entidad = fila["nombre_entidad"]
            if not entidad:
                entidad = etiqueta
        if entidad is None and not por_fila:
            anotar(cuadro, pagina["numero"], "fila de datos antes de la primera cabecera de ciudad o distrito", fila["rotulo"])
            continue
        valores, choques = asignar(fila, cortes_x, n)
        if len(choques) > 0:
            anotar(cuadro, pagina["numero"], "dos números en la misma columna: " + str(choques), fila["rotulo"])
        rotulo = etiqueta
        if por_fila:
            entidad = fila["nombre_entidad"]
            if not entidad:
                entidad = rotulo
            unidad = "entidad"
            rotulo = None
        else:
            unidad = "tipo_proceso"
        # "total" marca la fila de suma del cuadro (en la p. 131 todas son TOTAL NACIONAL y solo la última suma).
        if etiqueta.upper().startswith("TOTAL"):
            tipo_fila = "total"
        else:
            tipo_fila = "detalle"
        registro = {
            "fila": fila,
            "cuadro_origen": cuadro,
            "pagina_pdf": pagina["numero"],
            "firma": fmt["clave"],
            "familia": familia,
            "ambito": ambito,
            "titulo_pagina": pagina["titulo_pagina"],
            "entidad": entidad,
            "num_juzgados_pagina": juzgados,
            "unidad_fila": unidad,
            "grupo_proceso": None,
            "tipo_proceso": rotulo,
            "tipo_fila_derivado": tipo_fila,
            "orden_fila": len(salida) + len(bloque) + 1,
            "n_columnas": n,
        }
        for i in range(min(len(nombres), len(valores))):
            registro[nombres[i]] = valores[i]
        bloque.append(registro)
    cerrar_bloque(bloque, salida, pagina, x_grupo)
    for r in salida:
        r.pop("fila", None)
    return salida

## Extracción
Si una página resuelve otro número de columnas que su formato, se alinea contra la página de
referencia corrida lo que marque el desplazamiento.

In [ ]:
filas = []
for firma in formatos:
    fmt = formatos[firma]
    n = fmt["n_columnas"]
    declarado = LAYOUTS_PROCESOS.get(fmt["clave"])
    if declarado is None:
        familia = "sin_declarar"
        nombres = []
        for i in range(1, n + 1):
            nombres.append("col_" + str(i).zfill(2))
        anotar(",".join(fmt["cuadros"]), fmt["referencia"],
               "firma " + fmt["clave"] + " sin declarar en LAYOUTS_PROCESOS; " + str(n) + " columnas salen como col_NN")
    else:
        familia = declarado["familia"]
        nombres = declarado["columnas"]
        if len(nombres) != n:
            anotar(",".join(fmt["cuadros"]), fmt["referencia"],
                   "firma " + fmt["clave"] + ": se declararon " + str(len(nombres)) + " columnas y el PDF tiene " + str(n))
            continue
    ref = paginas_leidas[fmt["referencia"]]
    cortes_ref = cortes(ref["limites"])
    for p in sorted(fmt["paginas"]):
        pagina = paginas_leidas[p]
        cuadro = inv[p]
        if len(pagina["limites"]) == n:
            cortes_x = cortes(pagina["limites"])
        else:
            delta = desplazamiento(pagina, ref)
            cortes_x = []
            for c in cortes_ref:
                cortes_x.append(c + delta)
            anotar(cuadro, p, "la página resuelve " + str(len(pagina["limites"])) + " columnas y el formato tiene "
                   + str(n) + "; se alinea contra la página " + str(fmt["referencia"])
                   + " corrida {:+.1f} puntos".format(delta))
        filas = filas + filas_de_pagina(pagina, cuadro, fmt, familia, nombres, cortes_x, n)
print(len(filas), "filas de datos")
df = pd.DataFrame(filas)

# poppler >= 25 descarta la "A" superpuesta de "LA ACCIÓN" en el 5.3.2.1 (pp. 362-364) y deja
# "LA CCIÓN". Se restituye para obtener el mismo literal que con versiones anteriores.
roto = df["tipo_proceso"].astype(str).str.contains("HACIA LA CCIÓN PENAL", regex=False)
df.loc[roto, "tipo_proceso"] = df.loc[roto, "tipo_proceso"].str.replace("HACIA LA CCIÓN PENAL", "HACIA LA ACCIÓN PENAL", regex=False)
print("filas con la A restituida:", int(roto.sum()))

In [ ]:
for familia, parte in df.groupby("familia"):
    parte = parte.dropna(axis=1, how="all")
    guardar(parte, "crudo_procesos_" + familia + ".csv")
prob = pd.DataFrame(problemas)
guardar(prob, "problemas_extraccion_procesos.csv")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
por_familia = df["familia"].value_counts()
ejes[0].bar(por_familia.index, por_familia.values, color="tab:blue")
ejes[0].set_title("Filas extraídas por familia de cuadro")
columnas_por_firma = []
for firma in formatos:
    columnas_por_firma.append(formatos[firma]["n_columnas"])
ejes[1].hist(columnas_por_firma, bins=range(1, max(columnas_por_firma) + 2), color="tab:green")
ejes[1].set_title("Número de columnas de las 95 firmas")
ejes[1].set_xlabel("columnas")
plt.tight_layout()
plt.show()